# Solutions — HTTP and APIs

Only look here after you've actually tried the exercises in `http_api.ipynb`.

Every cell here runs offline against a mock response, exactly like the lesson.

### LESSON 45 — Exercise

**Part 1.**

In [ ]:
async function l45toResult(response) {
  if (!response.ok) {
    return { ok: false, error: `Request failed with status ${response.status}` };
  }

  // 1 - 204 means success with no body at all. Calling .json() would throw.
  if (response.status === 204) {
    return { ok: true, data: null };
  }

  const data = await response.json();

  // 2 - a successful response can still be the wrong shape.
  if (!Array.isArray(data)) {
    return { ok: false, error: "Expected a list" };
  }

  return { ok: true, data };
}

function l45mockResponse(status, body) {
  return {
    ok: status >= 200 && status < 300,
    status,
    json: async () => {
      if (status === 204) throw new SyntaxError("Unexpected end of JSON input");
      return body;
    },
  };
}

console.log("200 + array  ->", JSON.stringify(await l45toResult(l45mockResponse(200, [{ id: 1 }]))));
console.log("200 + object ->", JSON.stringify(await l45toResult(l45mockResponse(200, { id: 1 }))));
console.log("204 no body  ->", JSON.stringify(await l45toResult(l45mockResponse(204, null))));
console.log("404          ->", JSON.stringify(await l45toResult(l45mockResponse(404, { message: "Not Found" }))));

// The mock throws on .json() for a 204 on purpose - so if you had put the 204 check AFTER
// the parse, this cell would fail loudly instead of passing by luck. Order matters:
// ok -> 204 -> parse -> shape.

The ordering is the real content of Part 1. Every check has to happen before the thing it
protects against: the `ok` check before you trust the body, the 204 check before you parse,
and the shape check before you hand the data on.

**Part 2 — the playground.**

**1. Render first, request second.** The order is always:

```text
render — ... users:null ...
request — https://...
```

It cannot be the other way round. The request lives in an Effect, and Effects run at the end
of a commit (LESSON 39) — so the component must have rendered at least once before the request
can even start. That is why `users` needs a starting value your JSX can survive.

**2. The 404 in the Network tab.** The browser reports status **404**, and `fetch` reports
**success** — the promise resolved, no `catch` was reached by the network layer. The only
reason the console shows an error is the `if (!response.ok) throw` line. That is the JavaScript
course's warning with a React consequence attached.

**3. Deleting the `ok` check.** The 404's body (jsonplaceholder returns `{}`) parses fine, so
`data` becomes an empty object and `setUsers({})` puts it in state.

What then happens depends on your JSX. In this experiment the list renders nothing, the state
line reads as though a response arrived, and **no error is shown** — because no error ever
occurred as far as the code is concerned. On an API that returns a JSON error body with fields,
you would render those fields as if they were your data.

There is no crash and no console message. That is the whole danger: the failure mode is a
confidently wrong screen.

**Common mistakes.**

- Starting `users` at `null` and then writing `users.map(...)` with no guard. The first render
  happens before the data exists, every time.
- Putting the `204` check after `await response.json()`. The parse throws first.
- Treating a `catch` as "the request failed". It catches network failures and anything *you*
  threw — a 500 that you never checked for reaches neither.
- Fetching in the component body instead of an Effect. It would run during render, on every
  render, forever.

### LESSON 45 — Mini challenge

**The four problems.**

| | problem | fix, and where |
|---|---|---|
| 1 | **No `response.ok` check.** A 404 or 500 body is parsed and stored as if it were users. | LESSON 45 — check `ok` before trusting the body |
| 2 | **No cleanup and no `ignore` flag.** Change `url` quickly and an old response can arrive last and overwrite the new one. | LESSON 43 — `let ignore = false` plus a cleanup |
| 3 | **No error handling at all.** A rejected promise from a network failure becomes an unhandled rejection; the component sits there showing an empty list. | LESSON 46 |
| 4 | **No loading state.** Between the first render and the response the user sees an empty `<ul>` that looks like "no users" rather than "not yet". | LESSON 46 |

A fifth, if you spotted it, is legitimate: `users` starting as `[]` means `.map` is safe, which
is good — but it also means an empty result and a not-yet-loaded result look identical. That is
LESSON 46's distinction between *empty* and *loading*.

**Which a user notices first on a slow connection:** number 4. Everything else is invisible
until something goes wrong; a missing loading state is visible on every single load, and on a
slow connection it is the entire experience. The page appears to say "this user has no data"
for two seconds and then silently fills in.

**Which produces a wrong screen rather than a crash:** number 1, and it is worse for exactly
that reason. A crash is loud, immediate, and lands in your error tracking. A rendered error
body is silent: the user sees a plausible-looking page, nobody gets an alert, and the bug is
found weeks later by someone asking why a list is empty for one customer. Failures that look
like success are the expensive kind.

### LESSON 46 — Exercise

**Part 1.**

In [ ]:
function l46statusOf({ loading, error, data }) {
  const hasData = Array.isArray(data) && data.length > 0;

  // 1 - already showing something and fetching newer: that is not a blank "loading"
  if (loading && hasData) return "refreshing";
  if (loading) return "loading";

  // 2 - an error with data already on screen: show the stale data WITH a warning.
  //     Reasoning: throwing away good data the user is reading, to replace it with an error
  //     box, makes a recoverable network blip look like a broken app. Keep what works and
  //     say the refresh failed. If the data were something dangerous to act on while stale -
  //     a bank balance, a stock level - the opposite choice would be right, and that is a
  //     product decision, not a React one.
  if (error && hasData) return "stale-with-error";
  if (error) return "error";

  if (data === null || data === undefined) return "loading";
  if (Array.isArray(data) && data.length === 0) return "empty";
  return "success";
}

const l46six = [
  ["nothing yet         ", { loading: false, error: null, data: null }],
  ["first load          ", { loading: true, error: null, data: null }],
  ["failed first load   ", { loading: false, error: "500", data: null }],
  ["loaded, empty       ", { loading: false, error: null, data: [] }],
  ["loaded, with data   ", { loading: false, error: null, data: [{ id: 1 }] }],
  ["refreshing with data", { loading: true, error: null, data: [{ id: 1 }] }],
  ["refresh failed      ", { loading: false, error: "500", data: [{ id: 1 }] }],
];

for (const [label, state] of l46six) console.log(label, "->", l46statusOf(state));

Note the order inside the function. `refreshing` has to be checked before `loading`, and
`stale-with-error` before `error`, because the more specific case is a *combination* — test it
first or the general branch swallows it.

**Part 2 — the playground.**

**1. The empty result.** Not an error. The console shows `ok — 0 users` and no error line; the
screen says `0 users`, which is technically true and useless. It should say something a person
would write: "No users match." — a sentence, not a count.

**2. The empty-state branch.** Placed after loading and error, before the list:

```jsx
if (!loading && !error && users && users.length === 0) {
  return <p>No users match.</p>;
}
```

**3. Starting `users` at `[]`.** You can no longer distinguish **empty** from **not yet
loaded**. Both are an empty array with no error. On screen that shows up as the app confidently
saying "No users match." during the first load, before any response has arrived — telling the
user a fact you do not yet know.

**Common mistakes.**

- Treating an empty result as an error. A search with no matches is a successful search.
- Rendering `users.length === 0` before checking `loading`. The empty message flashes on every
  load.
- Showing a count instead of a sentence. "0 results" is a number; "No users match" is an answer.
- Adding a `hasLoaded` boolean *as well as* `loading` and `data`. Usually one of the three is
  derivable — LESSON 29 still applies.

### LESSON 46 — Mini challenge

**1. Three bugs, three confused pairs.**

- **empty confused with loading.** `invoices` starts at `[]`, the empty-state message renders
  immediately, and the request is still in flight. The user reads "no invoices" a second before
  the data arrives.
- **error confused with empty.** The request failed, `catch` runs, but the component only ever
  renders the list — which is still `[]` — so a 500 is displayed as "no invoices".
- **success confused with empty.** The `response.ok` check is missing, a 404 body is parsed
  into state, and `data.invoices` is `undefined`, so the list renders as empty. The request
  "succeeded" and the screen says there is nothing.

**2. What they should have seen:** a spinner or skeleton; an error message with a retry; and
their invoices.

**3. Impossible to diagnose from the screenshot: all three look identical**, and that is the
point of the challenge — but the one that is *genuinely* undiagnosable is the third, because
nothing went wrong from the app's perspective. The first two leave traces: a screenshot taken
a second later would show the data, and an error would usually appear in logs. A parsed error
body leaves no trace anywhere — no exception, no failed request in the Network tab's eyes, no
log line. You would have to reproduce the exact API failure to see it.

**4. Why a spinner fixes at most one.** A spinner only addresses the first bug — the one where
the app spoke too early. It does nothing about an error rendered as emptiness, and nothing
about a successful-but-wrong response. Reaching for a spinner treats the symptom ("the message
appeared too soon") rather than the cause ("the component cannot tell its four states apart").

### LESSON 47 — Exercise

In [ ]:
// 1. Measured, three presses, identical every time:
//
//      forced — caught AbortError: signal is aborted without reason
//
//    Write the condition against `name`. The name "AbortError" is specified; the message text
//    is not, and differs between browsers and versions. `e.name === "AbortError"` is stable,
//    `e.message.includes("aborted")` is a guess that will break.
//
// 2. Deleting `if (problem.name === "AbortError") return;` and reloading quickly puts
//    "AbortError: signal is aborted without reason" into `last error seen`. It is misleading
//    because NOTHING FAILED - the user sees an error caused entirely by the app's own cleanup
//    doing its job. The faster they interact, the more errors they get, which is the exact
//    opposite of how an app should behave.
//
// 3. Removing `controller.abort()` but keeping `ignore = true`: the request is still sent and
//    still completes - visible in the Network tab as requests that nobody is waiting for.
//    What IS protected is the state: `ignore` is true for that run, so the response cannot
//    call any setter. Correct screen, wasted work.
//
// 4. `ignore` alone is enough when the request is cheap and rare - a small payload on mount
//    that will not be superseded often. Leaving out `abort` matters when requests are
//    expensive or frequent: a search-as-you-type against a slow endpoint, a large download,
//    or a mobile connection where every abandoned request costs the user data and battery.

console.log("ignore protects the state; abort protects the work");

**Common mistakes.**

- Matching on the error message instead of the name.
- Aborting but not ignoring, then being surprised that a response applied in the gap.
- Putting `controller.abort()` anywhere other than the cleanup - aborting in the body cancels
  the request you just made.
- Treating every `catch` as something to show the user. Some of what lands there is you.

### LESSON 47 — Mini challenge

**A** — defects: no `response.ok` check, so an error body is parsed and stored; **and** the
catch calls `setError` on an `AbortError`, so every cleanup produces a spurious error. Result:
both a **wrong screen** (404 body rendered as data) and a **spurious error** (the app blaming
itself for its own cancellation).

**B** — defects: `[]` dependencies while the Effect reads `userId`, so it fetches the first
user and never refetches — the classic LESSON 40 drift, where the prop changes and the screen
does not. No `ok` check, no error handling at all, no abort. Result: a **wrong screen**, and a
completely silent one.

**C** — substantially correct. It checks `ok`, it handles `AbortError` by name, it aborts in
the cleanup, and its dependency array is right. Two nits rather than defects: `throw new
Error(r.status)` passes a number where a string is expected, so the message reads as a bare
`"404"`; and there is no `loading` state, so the four states of LESSON 46 are not all
representable.

**When C is safe without `ignore`, and what would break it.** It is safe because every request
it starts is aborted by the cleanup, and an aborted request rejects rather than resolving — so
a superseded response can never reach `setData`. The single change that would make it unsafe is
**removing the `signal`** (or calling a service that ignores it): the request would then
complete normally after the cleanup ran, `setData` would fire with stale data, and there would
be no flag to stop it. In other words `abort` is doing `ignore`'s job here, and only because it
is wired all the way through.

**Which would pass a LESSON 43-only review: B.** It has the inner-function-and-`ignore` shape
that L43 teaches, so it pattern-matches as correct — and its actual defect is the empty
dependency array, which is LESSON 40's material. A reviewer checking only for the race-condition
idiom would wave it through.

### LESSON 48 — Exercise

**Part 1.**

In [ ]:
function l48makeUserService(fetchImpl) {
  const BASE = "https://example.test";

  return {
    async getUsers({ signal } = {}) {
      const response = await fetchImpl(`${BASE}/users`, { signal });
      if (!response.ok) return { ok: false, error: `Request failed with status ${response.status}` };
      return { ok: true, data: await response.json() };
    },

    async getUser(id, { signal } = {}) {
      const response = await fetchImpl(`${BASE}/users/${encodeURIComponent(id)}`, { signal });
      if (response.status === 404) {
        return { ok: false, error: "User not found", notFound: true };
      }
      if (!response.ok) return { ok: false, error: `Request failed with status ${response.status}` };
      return { ok: true, data: await response.json() };
    },

    async searchUsers(query, { signal } = {}) {
      const url = `${BASE}/users?q=${encodeURIComponent(query)}`;
      const response = await fetchImpl(url, { signal });
      if (!response.ok) return { ok: false, error: `Request failed with status ${response.status}` };
      return { ok: true, data: await response.json() };
    },
  };
}

// Fakes that RECORD the url they were called with, so the encoding can be proved.
function l48spy(status, body) {
  const calls = [];
  const impl = async (url) => {
    calls.push(url);
    return { ok: status >= 200 && status < 300, status, json: async () => body };
  };
  return { impl, calls };
}

const l48found = l48spy(200, { id: 7, name: "Ada" });
console.log("found    ->", JSON.stringify(await l48makeUserService(l48found.impl).getUser(7)));

const l48missing = l48spy(404, {});
console.log("missing  ->", JSON.stringify(await l48makeUserService(l48missing.impl).getUser(99)));

const l48broken = l48spy(500, {});
console.log("server   ->", JSON.stringify(await l48makeUserService(l48broken.impl).getUser(1)));

const l48search = l48spy(200, []);
await l48makeUserService(l48search.impl).searchUsers("ada & co");
console.log("search url ->", l48search.calls[0]);
console.log("encoded?   ->", l48search.calls[0].includes("ada%20%26%20co") || l48search.calls[0].includes("ada+%26+co"));

// The spy is the point: the service is testable, offline, including the part that is easiest
// to get wrong. Without encodeURIComponent the "&" would start a new query parameter and the
// search would silently look for "ada ".

`notFound: true` is worth the extra field. The component needs to say "no such user" rather
than "something went wrong", and it must not have to parse an error string to find out.

**Part 2 — the sketch.**

In [ ]:
// src/services/employees.js
//    KNOWS:        the base URL, the paths, the response shape, how a failure becomes a result
//    MUST NOT KNOW: React, state, components, what a page looks like
//    exports:      getEmployees({signal}), getEmployee(id, {signal})
//
// src/components/EmployeeList.jsx
//    KNOWS:        the four states, what a list row looks like, which employee is selected
//    MUST NOT KNOW: any URL, any HTTP status code, the shape of the raw response
//
// src/components/EmployeeDetail.jsx
//    KNOWS:        how to render one employee, and what "not found" should look like
//    MUST NOT KNOW: how "not found" was determined
//
// The test for whether the line is drawn correctly: search the components folder for "http".
// If it finds anything, a URL has leaked upwards. Search the services folder for "useState".
// If it finds anything, React has leaked downwards.

console.log("services own the URL; components own the screen");

### LESSON 48 — Mini challenge

**A — React leaked downwards.** The service takes `setUsers` and `setError`, so it now depends
on a component's state. It cannot be called from anywhere else, cannot be tested without
inventing fake setters, and cannot serve two components with different state shapes. The
service should return a result and let the caller decide.

**B — the URL leaked upwards.** The component contains a full URL including query parameters.
The API version, the path and the filters are now spread through the UI, so moving the API
means grepping components. This is the case the JavaScript course's rule was written against.

**C — the response leaked upwards.** Returning the raw `Response` looks neutral but pushes the
work outward. See below.

**D — the view leaked downwards.** The service returns JSX. It now imports React, it decides
what a user looks like on screen, and it cannot be used by anything that renders differently —
a table, a dropdown, a CSV export. It is not a service; it is a component that also fetches.

**Why C is the interesting failure.** Returning the `Response` forces **every caller** to
duplicate the same three steps: check `response.ok`, decide what a failure means, and call
`await response.json()`. That is precisely the knowledge the service exists to own, so:

- the `ok` check will be forgotten in one of the five call sites, and that one will render an
  error body as data (LESSON 45);
- a status-code decision — is 404 "not found" or "broken"? — gets made differently in different
  components;
- and every component now knows what HTTP is, which was the thing being prevented.

A service that returns a `Response` has moved the file boundary without moving the
responsibility. The point of the split is not "fetch lives elsewhere"; it is that **the
component never learns what HTTP is**.